In [ ]:
import glob
import sys
from itertools import batched, chain
from pathlib import Path
from warnings import warn

from PIL import Image
from IPython.display import display, Markdown

In [ ]:
# Use this module without installing it.
sys.path.insert(0, "./src")
from jetpyck.gfxdat import *

# These are not exported by default:
from jetpyck.gfxdat import jswitch_name_decode, jswitch_name_encode

In [ ]:
BASEDIR = Path("./tilesets/")
GAMEDIR = Path("/home/denilson/Games/jetpack/")

In [ ]:
for f in sorted(GAMEDIR.glob("_jetp_?.dat", case_sensitive=False), key=lambda f: f.name.lower()):
    data = f.read_bytes()
    print("{:>12} {!r:36} = {!r}".format(f.name, jswitch_name_decode(data), data))

---

I could probably post this research to https://tcrf.net/ and/or https://datacrystal.tcrf.net/ and/or https://moddingwiki.shikadi.net/wiki/

But first I'll write everything down and save on GitHub.

---

In [ ]:
foo = JetpackGfx.load_from_filename(GAMEDIR / '_JETP_A0.DAT')

In [ ]:
foo.color_cycles

In [ ]:
for img in foo.get_images(zoom=2):
    display(img)

In [ ]:
foo

In [ ]:
foo._ipython_diplay_zoom = 3
foo

In [ ]:
experim = JetpackGfx.load_from_filename(GAMEDIR / '_JETP_X0.DAT')
experim._ipython_diplay_zoom = 3
experim

In [ ]:
with open(GAMEDIR / '_JETP_X.DAT', 'wb') as f:
    f.write(jswitch_name_encode('eXperimental tileset for reverse engineering'))

In [ ]:
for f in GAMEDIR.glob("_jetp_?0.dat", case_sensitive=False):
    display(JetpackGfx.load_from_bytes(f.read_bytes()))

In [ ]:
for filename in sorted(chain(
    glob.iglob("/home/denilson/Games/jetpack/JETPACK?.DAT"),
    glob.iglob("/home/denilson/Games/oldversions_jetpack/OLD_JET/JETPACK?.DAT"),
    # Disabled because these are the exact same graphics as the full version.
    #glob.iglob("/home/denilson/Games/shareware_jetpack/JETPACK?.DAT"),
    # Disabled because these have a different file format.
    #glob.iglob("/home/denilson/Games/xjetpack/JETPACK?.DAT"),
    #glob.iglob("/home/denilson/Games/jetpack/SQUAREZ/GRAPHICS.SQZ"),
    #glob.iglob("/home/denilson/Games/oldversions_jetpack/OLD_SQZ/SQUAREZ?.DAT"),
)):
    print(filename)
    with open(filename, "rb") as f:
        img = JetpackGfx.load_from_bytes(f.read())
        if "OLD_JET" in filename:
            img.color_cycles = []
        if "JETPACK1.DAT" in filename:
            img.color_cycles = [JetpackColorCycle.high_scores()]
        display(img)

In [ ]:
for filename in ["JETPACK0.DAT", "JETPACK1.DAT"]:
    pathname = GAMEDIR / filename
    gfx = JetpackGfx.load_from_filename(pathname)
    if pathname.name == "JETPACK1.DAT":
        gfx.color_cycles = [JetpackColorCycle.high_scores()]

    # Good static formats, preserving the palette:
    gfx.save_as_png(pathname.with_suffix(".png"))
    gfx.save_as_gif(pathname.with_suffix(".gif"))
    # Good animated formats, the palette is not preserved:
    gfx.save_as_gif_animated(pathname.with_suffix(".anim.gif"))
    gfx.save_as_webp_animated(pathname.with_suffix(".anim.webp"))

    # Modern format that does not preserve the palette:
    gfx.save_as_webp(pathname.with_suffix(".webp"))

    # Legacy formats (with large file sizes) that preserve the palette:
    gfx.save_as_bmp(pathname.with_suffix(".bmp"))
    gfx.save_as_pcx(pathname.with_suffix(".pcx"))

    # Checking if we can load a valid JetpackGfx object from the exported images,
    # and reporting if it is identical (lossless) to the original .DAT file.
    for imgfile in [pathname.with_suffix(ext) for ext in ['.png', '.gif', '.webp', '.bmp', '.pcx']]:
        msg = ""
        try:
            fromimg = JetpackGfx.load_from_image_file(imgfile)
            fromimg.color_cycles = gfx.color_cycles[:]
        except ValueError as e:
            fromimg = None
            msg = str(e)
        print(f"{pathname} {"==" if gfx == fromimg else "!="} {imgfile.name} {msg}")

In [ ]:
for pathname in sorted(Path(x) for x in chain(
    GAMEDIR.glob("_jetp_?0.dat", case_sensitive=False),
    GAMEDIR.glob("JETPACK?.DAT", case_sensitive=False),
    glob.iglob("/home/denilson/Games/oldversions_jetpack/OLD_JET/JETPACK?.DAT"),
)):
    srcdat = pathname.read_bytes()
    gfx = JetpackGfx.load_from_bytes(srcdat)
    dstdat = gfx.pack()
    print(f"{pathname} is {"equal" if srcdat == dstdat else "different"}")

In [ ]:
def explore_font_colors():
    ret = [
        '| gray | white | red | orange | green |',
        '| ---- | ----- | --- | ------ | ----- |',
    ]
    for graycolor in range(1, 32):
        white = (graycolor - 2) % 256
        if white == 0:
            white = 1
        red = graycolor // 2 + 31
        orange = red + 1*16
        green = red + 4*16
        ret.append(f'| {graycolor:3} | {white:3} | {red:3} | {orange:3} | {green:3} |')
    display(Markdown('\n'.join(ret)))

explore_font_colors()